<a href="https://colab.research.google.com/github/gastonm3112/cedears-rebalancing-tool/blob/main/src/Rebalanceador_Cedears_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 📝 CONFIGURACIÓN DEL REBALANCEO
# ==========================================
# @markdown ### 1. Datos de tu Archivo Google Sheets
NOMBRE_ARCHIVO_SHEETS = "mi_cartera" # @param {type:"string"}
PESTANA_DATOS = "cartera" # @param {type:"string"}

# @markdown ### 2. Gestión de Liquidez
# @markdown ¿Cuánto dinero nuevo vas a ingresar hoy a la cuenta? (En pesos)
MONTO_EXTRA_A_INYECTAR = 50000 # @param {type:"number"}

# ==========================================
# 🚀 INICIO DEL PROCESO (NO TOCAR ABAJO)
# ==========================================
import pandas as pd
import yfinance as yf
import gspread
from google.colab import auth
from google.auth import default
from gspread_dataframe import get_as_dataframe, set_with_dataframe
import datetime
import sys

def instalar_y_correr():
    print("⏳ Verificando librerías...")
    # Truco para instalar silenciosamente solo si falta
    try:
        import yfinance
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "yfinance", "gspread", "gspread-dataframe"])

    print("🔑 Autenticando con Google...")
    try:
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)
    except Exception as e:
        print(f"❌ Error de autenticación: {e}")
        return

    # --- 1. ABRIR HOJA ---
    print(f"📂 Abriendo archivo: '{NOMBRE_ARCHIVO_SHEETS}'...")
    try:
        sh = gc.open(NOMBRE_ARCHIVO_SHEETS)
        worksheet = sh.worksheet(PESTANA_DATOS)
    except Exception as e:
        print(f"❌ Error: No se encuentra el archivo o la pestaña. Verifica los nombres en el formulario.")
        return

    # --- 2. LEER DATOS ---
    df = get_as_dataframe(worksheet, evaluate_formulas=True)
    df = df.dropna(how='all')
    # Normalizar columnas
    df.columns = [str(c).lower().strip().replace(' ', '_') for c in df.columns]

    if 'ticker' not in df.columns or 'objetivo' not in df.columns:
        print("❌ Error: Faltan columnas 'Ticker' u 'Objetivo'.")
        return

    df = df[df['ticker'].notna()]
    lista_tickers = df['ticker'].astype(str).tolist()
    print(f"📊 Analizando cartera ({len(lista_tickers)} activos)...")

    # --- 3. OBTENER PRECIOS (Yahoo + Manual) ---
    tickers_ba = [t + ".BA" for t in lista_tickers]
    precios = {}

    # Intento descarga masiva
    try:
        datos_yahoo = yf.download(tickers_ba, period="1d", progress=False)['Close'].iloc[-1]
    except:
        datos_yahoo = pd.Series(dtype=float)

    for index, row in df.iterrows():
        t = str(row['ticker'])
        ticker_yahoo = t + ".BA"
        p_final = 0

        # A) Yahoo
        try:
            val = datos_yahoo.get(ticker_yahoo)
            if not pd.isna(val) and val > 0: p_final = val
        except: pass

        # B) Manual (Columna 'precio_manual' en el Excel)
        if p_final <= 0 and 'precio_manual' in df.columns:
            try:
                val_manual = float(row['precio_manual'])
                if val_manual > 0:
                    p_final = val_manual
                    print(f"   ℹ️ Usando precio manual para {t}: ${val_manual:,.2f}")
            except: pass

        precios[t] = p_final

    # --- 4. CÁLCULOS CON CASH EXTRA ---
    resultados = []
    total_tenencia_actual = 0

    # Valorizar
    for idx, row in df.iterrows():
        t = str(row['ticker'])
        try: nominales = float(row['nominales'])
        except: nominales = 0
        p = precios.get(t, 0)

        valor = nominales * p
        total_tenencia_actual += valor

        df.at[idx, '_p'] = p
        df.at[idx, '_val'] = valor

    # Lógica de Inyección
    total_cartera_futura = total_tenencia_actual + MONTO_EXTRA_A_INYECTAR

    print(f"\n💎 Valor Actual:  ${total_tenencia_actual:,.2f}")
    if MONTO_EXTRA_A_INYECTAR > 0:
        print(f"💵 Cash Nuevo:    ${MONTO_EXTRA_A_INYECTAR:,.2f}")
        print(f"🚀 Total Futuro:  ${total_cartera_futura:,.2f}")

    for idx, row in df.iterrows():
        t = str(row['ticker'])
        p = row['_p']
        val_ten = row['_val']
        try: obj = float(row['objetivo'])
        except: obj = 0

        # Cuánto debería tener considerando la plata nueva
        meta_dinero = total_cartera_futura * obj
        diferencia = meta_dinero - val_ten

        orden = 0
        if p > 0: orden = round(diferencia / p)

        accion = "MANTENER"
        if orden > 0: accion = "COMPRAR"
        elif orden < 0: accion = "VENDER"

        resultados.append({
            "Ticker": t,
            "Precio": p,
            "Tenencia ($)": val_ten,
            "Objetivo %": obj,
            "Diferencia ($)": diferencia,
            "ORDEN (Nominales)": int(orden),
            "ACCIÓN": accion
        })

    # --- 5. GUARDAR ---
    df_final = pd.DataFrame(resultados)
    nombre_pestana = f"Rebalanceo_{datetime.date.today()}"

    print(f"💾 Creando pestaña '{nombre_pestana}'...")
    try:
        try:
            ws_res = sh.add_worksheet(title=nombre_pestana, rows="60", cols="10")
        except:
            ws_res = sh.worksheet(nombre_pestana)
            ws_res.clear()

        set_with_dataframe(ws_res, df_final)
        ws_res.format('A1:G1', {'textFormat': {'bold': True}})
        print(f"✅ ¡LISTO! Revisa la hoja '{NOMBRE_ARCHIVO_SHEETS}'.")
    except Exception as e:
        print(f"❌ Error guardando: {e}")

# Ejecutar todo
instalar_y_correr()